## Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


## Import Package

In [ ]:
!pip install -q sentence-transformers

In [12]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Load Preprocessed Data

Reuses `df_complaints_categorized.csv` from `01_data_cleaning_and_preprocessing.ipynb` - already has `mk`/`md`/`yr`, `DATEA`, and `PARENT_GROUP`. Nothing from Part A's Option 1/Option 2 recall split is needed here - Part B isn't anchored to campaigns.

In [15]:
complaints_df = pd.read_csv('data/df_complaints_categorized.csv', low_memory=False)
complaints_df['DATEA'] = pd.to_datetime(complaints_df['DATEA'], errors='coerce')
print("Complaints:", complaints_df.shape)

Complaints: (2249816, 56)


## Severity & Equipment Flags

Direct per-complaint attributes (not aggregated across a pool, unlike Part A's pct_crash/pct_fire/pct_injured/pct_deaths) - each complaint already has its own raw value, just converted to a clean 0/1 scale.

- `crash_flag`: 1 if this complaint's `CRASH` field is `Y` (the incident involved a crash).
- `fire_flag`: 1 if this complaint's `FIRE` field is `Y` (the incident involved a fire).
- `injured_flag`: 1 if this complaint's `INJURED` count is greater than 0.
- `death_flag`: 1 if this complaint's `DEATHS` count is greater than 0.
- `police_report_flag`: 1 if a police report was filed (`POLICE_RPT_YN` = `Y`).
- `medical_attn_flag`: 1 if medical attention was sought (`MEDICAL_ATTN` = `Y`).
- `towed_flag`: 1 if the vehicle had to be towed (`VEHICLES_TOWED_YN` = `Y`).
- `anti_brakes_flag`: 1 if the vehicle has anti-lock brakes (`ANTI_BRAKES_YN` = `Y`) - an equipment-profile attribute, not a severity signal.
- `cruise_control_flag`: 1 if the vehicle has cruise control (`CRUISE_CONT_YN` = `Y`) - also equipment profile, not severity.


In [16]:
complaints_df['crash_flag'] = (complaints_df['CRASH'] == 'Y').astype(int)
complaints_df['fire_flag'] = (complaints_df['FIRE'] == 'Y').astype(int)
complaints_df['injured_flag'] = (pd.to_numeric(complaints_df['INJURED'], errors='coerce').fillna(0) > 0).astype(int)
complaints_df['death_flag'] = (pd.to_numeric(complaints_df['DEATHS'], errors='coerce').fillna(0) > 0).astype(int)
complaints_df['police_report_flag'] = (complaints_df['POLICE_RPT_YN'] == 'Y').astype(int)
complaints_df['medical_attn_flag'] = (complaints_df['MEDICAL_ATTN'] == 'Y').astype(int)
complaints_df['towed_flag'] = (complaints_df['VEHICLES_TOWED_YN'] == 'Y').astype(int)
complaints_df['anti_brakes_flag'] = (complaints_df['ANTI_BRAKES_YN'] == 'Y').astype(int)
complaints_df['cruise_control_flag'] = (complaints_df['CRUISE_CONT_YN'] == 'Y').astype(int)

flag_cols = ['crash_flag', 'fire_flag', 'injured_flag', 'death_flag',
             'police_report_flag', 'medical_attn_flag', 'towed_flag',
             'anti_brakes_flag', 'cruise_control_flag']
print(complaints_df[flag_cols].mean().rename('rate'))


crash_flag             0.062407
fire_flag              0.025279
injured_flag           0.039878
death_flag             0.001945
police_report_flag     0.041171
medical_attn_flag      0.011952
towed_flag             0.025893
anti_brakes_flag       0.150931
cruise_control_flag    0.156248
Name: rate, dtype: float64


## Build Sentence Embeddings (per complaint)

Primary clustering input - one embedding per complaint's own `CDESCR` narrative, not glued/pooled like Part A's campaign-level text. Benchmarked at ~914 complaints/sec batched on CPU, so embedding the full set (~2.0M rows) takes roughly 35-40 minutes - no sampling needed for this step. (Sampling may still be needed for the clustering algorithms themselves later, in `03_unsupervised_clustering.ipynb` - that's a separate decision, not required here.)

In [17]:
texts = complaints_df['CDESCR'].fillna('no narrative').astype(str).tolist()
embeddings = embedding_model.encode(texts, batch_size=64, show_progress_bar=True)

emb_cols = [f'emb_{i}' for i in range(embeddings.shape[1])]
emb_df = pd.DataFrame(embeddings, columns=emb_cols, index=complaints_df.index)
print("Embeddings:", emb_df.shape)


Batches:   0%|          | 0/35154 [00:00<?, ?it/s]

Embeddings: (2249816, 384)


## Assemble Final Feature Table

Column-by-column breakdown of the final saved table:

**Clustering input (X) - fed into K-means/HDBSCAN/Agglomerative:**
- `emb_0` ... `emb_383`: the 384-dimension sentence embedding of this complaint's own narrative (`CDESCR`).
- `crash_flag`, `fire_flag`, `injured_flag`, `death_flag`, `police_report_flag`, `medical_attn_flag`, `towed_flag`: severity-related flags (see above).
- `anti_brakes_flag`, `cruise_control_flag`: equipment-profile flags (see above).

**For description only, after clustering - never fed into the clustering itself:**
- `mk`: the complaint's make, uppercased (e.g. `FORD`).
- `md`: the complaint's model, uppercased.
- `yr`: the complaint's model year.
- `DATEA`: the date this complaint was filed - used later to apply a follow-up window when checking recall rate per cluster.
- `PARENT_GROUP`: this complaint's own component category (the same 16-category scheme used for Part A's `y`) - used only to describe what's inside a cluster afterward. Kept out of the clustering input itself, along with `mk`/`md`/`yr`, so clusters aren't just trivially rediscovering the known categories or car brands instead of finding something new.


In [18]:
features_unsupervised = pd.concat([complaints_df[['mk', 'md', 'yr', 'DATEA', 'PARENT_GROUP'] + flag_cols], emb_df], axis=1,)
print("Final feature table:", features_unsupervised.shape)
features_unsupervised.head()

Final feature table: (2249816, 398)


,mk,md,yr,DATEA,PARENT_GROUP,crash_flag,fire_flag,injured_flag,death_flag,police_report_flag,...,emb_374,emb_375,emb_376,emb_377,emb_378,emb_379,emb_380,emb_381,emb_382,emb_383
0,VOLVO,760,1987,1995-01-03,Engine,0,0,0,0,0,...,-0.118114,-0.112822,0.026545,-0.074040,-0.072572,-0.003037,0.018568,0.048434,0.008437,-0.004528
1,FORD,THUNDERBIRD,1992,1995-01-03,Fuel,0,0,0,0,0,...,0.049502,-0.110971,-0.084964,0.031852,0.032895,-0.017793,0.041318,0.064131,0.020167,0.006246
2,KIA,SEPHIA,1994,1995-01-03,Transmission,1,0,0,0,0,...,0.022115,0.018701,-0.027887,-0.024169,-0.086984,0.024493,-0.029590,0.044365,-0.006677,-0.037692
3,DODGE,600,1987,1995-01-03,Fuel,0,0,0,0,0,...,0.042924,-0.001707,0.048698,0.005044,-0.006620,-0.019036,-0.008690,0.078550,0.050660,0.005729
4,DODGE,CARAVAN,1991,1995-01-03,Seating,0,0,0,0,0,...,-0.028921,0.038645,-0.029730,0.021788,0.009192,0.034593,-0.026567,0.071601,0.031950,0.018946


## Save Feature Engineering Result

Saved as Parquet, not CSV - at ~2M rows x ~392 numeric columns, a CSV would be enormous and slow; Parquet is far more compact and much faster to read back.

In [19]:
features_unsupervised.to_parquet('data/features_unsupervised.parquet')
print("Saved:", features_unsupervised.shape)

Saved: (2249816, 398)
